# OptiStop -- Exploratory Data Analysis

This notebook explores the synthetic commuter demand dataset generated by
`src/data_loader.py` before any clustering is applied. It covers:

1. Loading the dataset
2. Summary statistics
3. Distribution of demand weight
4. Geographic distribution of commuters
5. Demand by source hotspot
6. Correlation / density checks

Run `python main.py` once first (or just the cells below) to ensure
`data/synthetic_commuter_demand.csv` exists.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CONFIG
from src.data_loader import get_or_generate_demand_data

sns.set_theme(style="whitegrid")
%matplotlib inline


## 1. Load Data

In [ ]:
df = get_or_generate_demand_data(CONFIG)
print(df.shape)
df.head()


## 2. Summary Statistics

In [ ]:
df.describe()


In [ ]:
df.isnull().sum()


## 3. Distribution of Demand Weight

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["demand_weight"], bins=40, kde=True, ax=ax[0], color="#2563eb")
ax[0].set_title("Demand Weight Distribution")

sns.boxplot(x=df["demand_weight"], ax=ax[1], color="#16a34a")
ax[1].set_title("Demand Weight Boxplot")
plt.tight_layout()
plt.show()


## 4. Geographic Distribution of Commuters

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(df["longitude"], df["latitude"], c=df["demand_weight"],
                cmap="rocket_r", s=8, alpha=0.6)
plt.colorbar(sc, label="Demand weight")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title(f"Synthetic Commuter Demand -- {CONFIG.city_name}")
plt.show()


## 5. Demand by Source Hotspot

In [ ]:
hotspot_stats = (
    df.groupby("source_hotspot")
    .agg(n_commuters=("demand_weight", "count"), total_demand=("demand_weight", "sum"),
         avg_demand=("demand_weight", "mean"))
    .sort_values("total_demand", ascending=False)
)
hotspot_stats


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
hotspot_stats["total_demand"].plot(kind="bar", ax=ax, color="#dc2626")
ax.set_ylabel("Total demand")
ax.set_title("Total Commuter Demand by Hotspot")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Density Heatmap (KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
sns.kdeplot(x=df["longitude"], y=df["latitude"], weights=df["demand_weight"],
            fill=True, cmap="mako", thresh=0.02, levels=50, ax=ax)
ax.set_title("Commuter Demand Density (KDE)")
plt.show()


## Next Steps

With the demand distribution understood, proceed to `main.py` (or
`src/clustering.py`) to run K-Means clustering, select the optimal number
of clusters via the Elbow Method + Silhouette Score, and generate the
final stop recommendations.
